# 03 — N₂ Dissociation Curve

N₂ dissociation is a classic test of multireference methods:
- Near equilibrium: CCSD performs well.
- Near dissociation: strong correlation breaks CCSD; FCI is required.
- VQE-UCCSD and ADAPT-VQE should track FCI better than CCSD at long bonds.

**Active space**: 6 electrons in 6 orbitals (valence π + σ), 6-31G basis.


In [ ]:
import sys, os, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings('ignore')

pkg_root = Path.cwd().parent
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

import quantum_chem_bench.classical_solvers  # noqa
import quantum_chem_bench.quantum_solvers    # noqa

from quantum_chem_bench.core.config import BenchConfig
from quantum_chem_bench.analysis.benchmark import BenchmarkPlotter
from quantum_chem_bench.analysis.pes_scanner import PESScanner

In [ ]:
config = BenchConfig.from_yaml('../configs/n2_631g.yaml')

# CI mode: fast scan
if os.environ.get('CI_FAST_NB'):
    config.solvers.quantum = ['vqe_uccsd']
    config.solvers.classical = ['hf', 'ccsd', 'fci']
    distances = np.linspace(0.9, 2.0, 4).tolist()
else:
    distances = np.linspace(0.9, 2.5, 16).tolist()

print(f"Scanning {len(distances)} geometries: {distances[0]:.2f} — {distances[-1]:.2f} Å")
print(f"Solvers: {config.solvers.classical + config.solvers.quantum}")

In [ ]:
scanner = PESScanner(config, verbose=True)
pes = scanner.scan_bond(
    atom_indices=(0, 1),
    distances=distances,
    atom_symbols=('N', 'N'),
)
print(f"\nCompleted PES scan for methods: {list(pes['energies'].keys())}")

## Potential Energy Surface

In [ ]:
%matplotlib inline
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

plotter = BenchmarkPlotter(figsize=(10, 6))
fig = plotter.multi_method_pes(
    geometries=pes['geometries'],
    results_by_method=pes['energies'],
    x_label='N-N Distance (Å)',
    y_label='Energy (Ha)',
    title='N₂ Dissociation Curve (6-31G, 6e/6o active space)',
)
fig.savefig('n2_pes.png', dpi=150)
fig

## CCSD vs FCI error along the curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

energies = pes['energies']
geoms = pes['geometries']

if 'FCI' in energies:
    fig, ax = plt.subplots(figsize=(9, 4))
    fci_e = np.array(energies['FCI'])

    for method, color in [('CCSD', '#F44336'), ('VQE-UCCSD', '#2196F3'),
                           ('ADAPT-VQE', '#4CAF50')]:
        key = next((k for k in energies if method in k), None)
        if key:
            err = (np.array(energies[key]) - fci_e) * 1000
            ax.plot(geoms, err, 'o-', label=method, color=color)

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(1.6, color='gray', linewidth=0.8, linestyle=':', label='Chemical accuracy (1.6 mHa)')
    ax.set_xlabel('N-N Distance (Å)', fontsize=12)
    ax.set_ylabel('Error vs FCI (mHa)', fontsize=12)
    ax.set_title('N₂ — Method Accuracy vs FCI Along Dissociation', fontsize=12)
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig('n2_error_vs_fci.png', dpi=150)
    plt.show()
else:
    print('FCI not computed (too expensive for this active space). Add fci to classical solvers.')

## Wall-time comparison

Classical methods are faster but lose accuracy at dissociation.

In [ ]:
import pandas as pd

time_data = []
for method, times in pes['wall_times'].items():
    time_data.append({'Method': method, 'Total Time (s)': sum(times),
                      'Per Point (s)': sum(times)/len(times)})

df = pd.DataFrame(time_data).sort_values('Total Time (s)')
df